# Processing raw xenium into zarr
This notebook will process the raw xenium data into zarr format. It requires:

- cell_feature_matrix.zarr.zip
- transcripts_zarr, which is unzipped from transcripts.zarr.zip
- gene_panel.json
- cells.zarr.zip

all to be placed in the same directory as this notebook.

In [2]:
from scipy.sparse import csr_matrix
import numpy as np
import json
import pandas as pd
import anndata as ad
import zarr
import geopandas as gpd
from tifffile import imwrite
import rasterio
from shapely.geometry import Polygon
from rasterio.features import rasterize

First, we will read in several of the raw Xenium zipped outputs, including the cell features (transcripts summarised by cell centroid), as well as the gene panel to get the target names. 

In [3]:
z = zarr.open("cell_feature_matrix.zarr.zip", mode="r")

cf = z["cell_features"]

tz = zarr.open("transcripts_zarr/", mode="r")

with open("gene_panel.json") as f:
    panel = json.load(f)

# Get all the gene names
targets = panel["payload"]["targets"]

We will then define the genes names to retain from the panel JSON, to use as the Anndata var names. 

In [4]:
remove = ['Probe', 'Deprecated', 'Intergenic']
genes = [gene['type']['data']['name'] for gene in panel["payload"]['targets']]
genes = [str(gene) for gene in genes if not any(str(ignore) in gene for ignore in remove)]

Next, we will construct the Anndata.X slot containing the cell-level transcript/probe counts, and insert the gene names. 

In [5]:
z_cells = zarr.open("cells.zarr.zip", mode="r")  # or whatever path
cell_summary = z_cells["cell_summary"][:]       # shape (n_cells, 8)
cell_ids = z_cells["cell_id"][:, 0]

centroids = cell_summary[:, :2]  # shape (n_cells, 2)
X = csr_matrix(
    (
        cf["data"][:],
        cf["indices"][:],
        cf["indptr"][:],
    ),
    shape=(10030, len(centroids))   # cells x genes (indptr len - 1, inferred genes)
).T


adata = ad.AnnData(X = X[:,0:5001], obs = pd.DataFrame({
    "region": ["xenium_1"] * len(centroids)}), obsm = {'spatial': centroids})
adata.var_names = genes

adata.write_h5ad("./xenium_expression.h5ad")

/home/admin/.local/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Lastly, we will extract the single cell segmentation boundaries from the polygon sets, and rasterise them into a numpy mask array that can be imported into Rakaia. 

In [6]:
poly = z_cells["polygon_sets"]["1"]

cell_index = poly["cell_index"][:]
num_vertices = poly["num_vertices"][:]
vertices = poly["vertices"][:]


polygons = []
cell_ids = []

for i in range(len(cell_index)):
    nv = num_vertices[i]

    # skip invalid polygons
    if nv < 3:
        continue

    coords = vertices[i, :2 * nv].reshape(nv, 2)

    # shapely expects (x, y)
    poly_geom = Polygon(coords)

    # optional: fix invalid polygons
    if not poly_geom.is_valid:
        poly_geom = poly_geom.buffer(0)

    polygons.append(poly_geom)
    cell_ids.append(cell_index[i])

gdf = gpd.GeoDataFrame(
    geometry=polygons)

# get the int bounds to know where the segmentation mask is
x_min, y_min = np.min((adata.obsm['spatial']), axis=0)
x_max, y_max = np.max((adata.obsm['spatial']), axis=0)

transform = rasterio.transform.from_bounds(x_min, y_min, x_max,
                        y_max, int(x_max - x_min), int(y_max - y_min))

# IMP: this can cause gaps or missing cells if one cell completely engulfs another during sequential burn-in
shapes = [(geom, idx + 1) for idx, geom in enumerate(gdf.geometry)]

# Set the mask shape to be the same as the transcript bounds in rakaia
mask = rasterize(shapes=shapes, out_shape=(int(y_max - y_min), int(x_max - x_min)),
                             transform=transform, dtype='int32')

mask = np.flip(mask, axis=0)

imwrite("xenium_mask.tiff", mask.astype(np.uint32), photometric='minisblack')


We now have `xenium_expression.h5ad` and `xenium_mask.tiff` created in this directory, which are required inputs for the `xenium_imc_multimodal_integration.ipynb` notebook. 